Section 1 — Build DimProduct

In [1]:
#01 — Import library
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

np.random.seed(42)

In [2]:
#02 — Load the Feature Engineering Dataset
df = pd.read_csv(
    "../data/processed/transactions_enriched.csv",
    parse_dates=["InvoiceDate"]
)

df.head()

C:\Users\محمد الرويلي\AppData\Local\Temp\ipykernel_19284\1871355037.py:2: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SalesAmount,CostRatio,UnitCost,TotalCost,Profit,ProfitMargin,InvoiceDateOnly,Year,Quarter,Month,MonthName,MonthShort,Week,Day,DayName,DayOfWeek,IsWeekend,Season,YearMonth,DiscountPercent,PromotionApplied,PromotionType,OrderSize,SalesChannel,PaymentMethod,ShippingPriority
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,0.612362,1.56,9.36,5.94,38.82,2010-12-01,2010,Q4,12,December,Dec,48,1,Wednesday,2,False,Winter,2010-12,0,False,No Promotion,Small,Online,Credit Card,Standard
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0.785214,2.66,15.96,4.38,21.53,2010-12-01,2010,Q4,12,December,Dec,48,1,Wednesday,2,False,Winter,2010-12,15,True,Flash Sale,Small,In-Store,Cash,Standard
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,0.719598,1.98,15.84,6.16,28.00,2010-12-01,2010,Q4,12,December,Dec,48,1,Wednesday,2,False,Winter,2010-12,5,True,Member Discount,Small,Online,PayPal,Same Day
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0.679598,2.30,13.80,6.54,32.15,2010-12-01,2010,Q4,12,December,Dec,48,1,Wednesday,2,False,Winter,2010-12,0,False,No Promotion,Small,In-Store,Credit Card,Same Day
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0.546806,1.85,11.10,9.24,45.43,2010-12-01,2010,Q4,12,December,Dec,48,1,Wednesday,2,False,Winter,2010-12,0,False,No Promotion,Small,In-Store,Credit Card,Express


In [3]:
#03 — Extract Unique Products
dim_product = (
    df[
        [
            "StockCode",
            "Description"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

#Check the result
dim_product.head()

,StockCode,Description
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN
2,84406B,CREAM CUPID HEARTS COAT HANGER
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,84029E,RED WOOLLY HOTTIE WHITE HEART.


In [4]:
#04 — Generate ProductKey
dim_product.insert(
    0,
    "ProductKey",
    range(1, len(dim_product) + 1)
)

In [5]:
#05 — Verify Uniqueness
print("Unique Products :", dim_product["StockCode"].nunique())

print("Rows :", len(dim_product))

Unique Products : 3922
Rows : 4161


In [6]:
#06 — Check the Number of Products
print(f"Total Products: {len(dim_product):,}")

Total Products: 4,161


In [7]:
#07 — Save a Temporary Version
dim_product.to_csv(
    "../data/warehouse/DimProduct.csv",
    index=False
)

In [8]:
#08 — Create the category
#just cheaking
import sys
from pathlib import Path

# Get the project root directory
project_root = Path.cwd().parent

# Add it to Python's search path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

#import it 
import config.product_rules as pr

print(dir(pr))

['CATEGORY_RULES', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [9]:
#09 - import the CATEGORY_RULES 
from config.product_rules import CATEGORY_RULES

In [10]:
#10 — Create the categorization function
def categorize_product(description):
    description = str(description).upper()

    for category, keywords in pr.CATEGORY_RULES.items():
        for keyword in keywords:
            if keyword in description:
                return category

    return "Other"


In [11]:
#11 — Apply it
dim_product["Category"] = dim_product["Description"].apply(categorize_product)

In [12]:
#12 — Check the results
dim_product["Category"].value_counts()

Category
Home & Living            1806
Other                     599
Kitchen & Dining          475
Gifts & Accessories       338
Stationery                269
Fashion & Accessories     264
Seasonal                  177
Garden & Outdoor           97
Toys & Kids                76
Furniture                  41
Party Supplies             19
Name: count, dtype: int64

In [13]:
#13 — Inspect "Other"
other_products = (
    dim_product[dim_product["Category"] == "Other"]
    .sort_values("Description")
)

other_products.head(100)

,ProductKey,StockCode,Description,Category
3849,3850,23391,I LOVE LONDON MINI BACKPACK,Other
3976,3977,23391,I LOVE LONDON MINI RUCKSACK,Other
3808,3809,23411,TRELLIS COAT RACK,Other
2031,2032,22282,12 EGG HOUSE PAINTED WOOD,Other
4126,4127,23442,12 HANGING EGGS HAND PAINTED,Other
...,...,...,...,...
1884,1885,21468,BUTTERFLY CROCHET FOOD COVER,Other
2413,2414,90174,BUTTERFLY HAIR BAND,Other
1534,1535,BANK CHARGES,Bank Charges,Other
2770,2771,DCGS0070,CAMOUFLAGE DOG COLLAR,Other


In [14]:
#14 - build the hierarchy
#Step 1 — Create Department
DEPARTMENT_MAP = {
    "Home & Living": "Home",
    "Kitchen & Dining": "Home",
    "Furniture": "Home",

    "Seasonal": "Seasonal",

    "Gifts & Accessories": "Accessories",
    "Fashion & Accessories": "Accessories",

    "Stationery": "Office",

    "Toys & Kids": "Kids",

    "Garden & Outdoor": "Outdoor",

    "Party Supplies": "Party",

    "Other": "Other"
}

In [15]:
dim_product["Department"] = dim_product["Category"].map(DEPARTMENT_MAP)

In [16]:
#Step 2 — Create Subcategory
SUBCATEGORY_RULES = {
    "Candle Holders": ["CANDLE", "T-LIGHT"],
    "Photo Frames": ["FRAME", "PHOTO"],
    "Storage": ["BOX", "TIN", "BASKET", "JAR", "TRAY"],
    "Kitchenware": ["MUG", "CUP", "BOWL", "PLATE", "DISH"],
    "Glassware": ["GLASS", "WINE", "CHAMPAGNE"],
    "Bags": ["BAG"],
    "Jewelry": ["RING", "BRACELET", "NECKLACE"],
    "Christmas": ["CHRISTMAS", "XMAS", "SANTA"],
    "Garden": ["GARDEN", "FLOWER", "PLANT"],
    "Stationery": ["CARD", "NOTEBOOK", "PAPER"],
    "Toys": ["TOY", "TEDDY", "DOLL"],
    "Furniture": ["CHAIR", "TABLE", "STOOL", "CABINET"]
}

In [17]:
def get_subcategory(description):
    description = str(description).upper()

    for subcategory, keywords in SUBCATEGORY_RULES.items():
        for keyword in keywords:
            if keyword in description:
                return subcategory

    return "Other"

dim_product["Subcategory"] = dim_product["Description"].apply(get_subcategory)

In [18]:
#Step 3 — Reorder the Product Dimension
dim_product = dim_product[
    [
        "ProductKey",
        "StockCode",
        "Description",
        "Department",
        "Category",
        "Subcategory"
    ]
]

In [19]:
#Validate
dim_product.head()

,ProductKey,StockCode,Description,Department,Category,Subcategory
0,1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,Home,Home & Living,Candle Holders
1,2,71053,WHITE METAL LANTERN,Home,Home & Living,Other
2,3,84406B,CREAM CUPID HEARTS COAT HANGER,Home,Home & Living,Kitchenware
3,4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,Home,Kitchen & Dining,Other
4,5,84029E,RED WOOLLY HOTTIE WHITE HEART.,Home,Home & Living,Other


In [20]:
#15 — Save the Dimension
dim_product.to_csv(
    "../data/warehouse/DimProduct.csv",
    index=False
)

Section 2 — DimCustomer

In [21]:
#01 — Load the Feature Engineering Dataset
import pandas as pd

df = pd.read_csv("../data/processed/transactions_enriched.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

C:\Users\محمد الرويلي\AppData\Local\Temp\ipykernel_19284\1191469688.py:4: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/transactions_enriched.csv")


In [22]:
#02 — replace Unknown Customers with 0
df["CustomerID"] = (
    df["CustomerID"]
    .fillna(0)
    .astype(int)
)

In [23]:
#03 — Create Customer Aggregation
dim_customer = (
    df.groupby("CustomerID")
      .agg(
          Country=("Country", "first"),

          FirstPurchaseDate=("InvoiceDate", "min"),
          LastPurchaseDate=("InvoiceDate", "max"),

          TotalOrders=("InvoiceNo", "nunique"),

          TotalSales=("SalesAmount", "sum"),

          TotalQuantity=("Quantity", "sum")
      )
      .reset_index()
)

In [24]:
#04 — Create CustomerKey
dim_customer.insert(0, "CustomerKey", range(1, len(dim_customer) + 1))

In [25]:
#05 — Average Order Value
dim_customer["AverageOrderValue"] = (
    dim_customer["TotalSales"] /
    dim_customer["TotalOrders"]
).round(2)

In [26]:
#06 — Customer Age
reference_date = df["InvoiceDate"].max()

dim_customer["CustomerAgeDays"] = (
    reference_date -
    dim_customer["FirstPurchaseDate"]
).dt.days

In [27]:
#07 — Check
dim_customer.head()

,CustomerKey,CustomerID,Country,FirstPurchaseDate,LastPurchaseDate,TotalOrders,TotalSales,TotalQuantity,AverageOrderValue,CustomerAgeDays
0,1,0,United Kingdom,2010-12-01 14:32:00,2011-12-09 10:26:00,1429,1754901.91,420418,1228.06,372
1,2,12346,United Kingdom,2011-01-18 10:01:00,2011-01-18 10:01:00,1,77183.60,74215,77183.60,325
2,3,12347,Iceland,2010-12-07 14:57:00,2011-12-07 15:52:00,7,4310.00,2458,615.71,366
3,4,12348,Finland,2010-12-16 19:09:00,2011-09-25 13:13:00,4,1797.24,2341,449.31,357
4,5,12349,Italy,2011-11-21 09:51:00,2011-11-21 09:51:00,1,1757.55,631,1757.55,18


In [28]:
#08 — Calculate Recency
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

dim_customer["Recency"] = (
    reference_date - dim_customer["LastPurchaseDate"]
).dt.days

In [29]:
#09 — Frequency
dim_customer["Frequency"] = dim_customer["TotalOrders"]

In [30]:
#10 — Monetary
dim_customer["Monetary"] = (
    dim_customer["TotalSales"]
    .round(2)
)

In [31]:
#11 — Create RFM Scores

#Recency Score
dim_customer["R_Score"] = pd.qcut(
    dim_customer["Recency"],
    4,
    labels=[4,3,2,1]
).astype(int)

#Frequency Score
dim_customer["F_Score"] = pd.qcut(
    dim_customer["Frequency"].rank(method="first"),
    4,
    labels=[1,2,3,4]
).astype(int)

#Monetary Score
dim_customer["M_Score"] = pd.qcut(
    dim_customer["Monetary"].rank(method="first"),
    4,
    labels=[1,2,3,4]
).astype(int)

In [32]:
#12 — Create the RFM Score
dim_customer["RFMScore"] = (
    dim_customer["R_Score"].astype(str) +
    dim_customer["F_Score"].astype(str) +
    dim_customer["M_Score"].astype(str)
)

In [33]:
#13 — Customer Segment (Business Rules)

conditions = [
    dim_customer["Monetary"] < 500,
    (dim_customer["Monetary"] >= 500) & (dim_customer["Monetary"] < 2000),
    (dim_customer["Monetary"] >= 2000) & (dim_customer["Monetary"] < 6000),
    dim_customer["Monetary"] >= 6000
]

labels = [
    "Bronze",
    "Silver",
    "Gold",
    "Platinum"
]

dim_customer["CustomerSegment"] = np.select(
    conditions,
    labels,
    default="Bronze"
)

In [34]:
dim_customer["CustomerSegment"].value_counts()

CustomerSegment
Bronze      1764
Silver      1678
Gold         687
Platinum     210
Name: count, dtype: int64

In [35]:
(
    dim_customer["CustomerSegment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

CustomerSegment
Bronze      40.7
Silver      38.7
Gold        15.8
Platinum     4.8
Name: proportion, dtype: float64

In [36]:
#14 — Loyalty Level
def loyalty_level(score):

    score = int(score)

    if score >= 444:
        return "VIP"

    elif score >= 344:
        return "Loyal"

    elif score >= 233:
        return "Regular"

    return "New"

#Apply it
dim_customer["LoyaltyLevel"] = (
    dim_customer["RFMScore"]
    .apply(loyalty_level)
)

In [37]:
#15 — Reorder the Columns
dim_customer = dim_customer[
    [
        "CustomerKey",
        "CustomerID",
        "Country",
        "FirstPurchaseDate",
        "LastPurchaseDate",
        "CustomerAgeDays",
        "TotalOrders",
        "TotalSales",
        "TotalQuantity",
        "AverageOrderValue",
        "Recency",
        "Frequency",
        "Monetary",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFMScore",
        "CustomerSegment",
        "LoyaltyLevel"
    ]
]

In [38]:
#16 — check
dim_customer.head(10)

,CustomerKey,CustomerID,Country,FirstPurchaseDate,LastPurchaseDate,CustomerAgeDays,TotalOrders,TotalSales,TotalQuantity,AverageOrderValue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFMScore,CustomerSegment,LoyaltyLevel
0,1,0,United Kingdom,2010-12-01 14:32:00,2011-12-09 10:26:00,372,1429,1754901.91,420418,1228.06,1,1429,1754901.91,4,4,4,444,Platinum,VIP
1,2,12346,United Kingdom,2011-01-18 10:01:00,2011-01-18 10:01:00,325,1,77183.60,74215,77183.60,326,1,77183.60,1,1,4,114,Platinum,New
2,3,12347,Iceland,2010-12-07 14:57:00,2011-12-07 15:52:00,366,7,4310.00,2458,615.71,2,7,4310.00,4,4,4,444,Gold,VIP
3,4,12348,Finland,2010-12-16 19:09:00,2011-09-25 13:13:00,357,4,1797.24,2341,449.31,75,4,1797.24,2,3,4,234,Silver,Regular
4,5,12349,Italy,2011-11-21 09:51:00,2011-11-21 09:51:00,18,1,1757.55,631,1757.55,19,1,1757.55,3,1,4,314,Silver,Regular
5,6,12350,Norway,2011-02-02 16:01:00,2011-02-02 16:01:00,309,1,334.40,197,334.40,310,1,334.40,1,1,2,112,Bronze,New
6,7,12352,Norway,2011-02-16 12:33:00,2011-11-03 14:37:00,296,8,2506.04,536,313.26,36,8,2506.04,3,4,4,344,Gold,Loyal
7,8,12353,Bahrain,2011-05-19 17:47:00,2011-05-19 17:47:00,203,1,89.00,20,89.00,204,1,89.00,1,1,1,111,Bronze,New
8,9,12354,Spain,2011-04-21 13:11:00,2011-04-21 13:11:00,231,1,1079.40,530,1079.40,232,1,1079.40,1,1,3,113,Silver,New
9,10,12355,Bahrain,2011-05-09 13:49:00,2011-05-09 13:49:00,213,1,459.40,240,459.40,214,1,459.40,1,1,2,112,Bronze,New


In [39]:
dim_customer["Monetary"].describe()

count    4.339000e+03
mean     2.452664e+03
std      2.808606e+04
min      3.750000e+00
25%      3.065050e+02
50%      6.685800e+02
75%      1.660890e+03
max      1.754902e+06
Name: Monetary, dtype: float64

In [40]:
dim_customer["Monetary"].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

0.25      306.5050
0.50      668.5800
0.75     1660.8900
0.90     3641.2560
0.95     5838.7310
0.99    20090.5652
Name: Monetary, dtype: float64

In [41]:
#17 - save Dimension
dim_customer.to_csv(
    "../data/warehouse/DimCustomer.csv",
    index=False
)

Section 3 — DimDate

In [42]:
#01 — Load the Feature Engineering Dataset
import pandas as pd

df = pd.read_csv("../data/processed/transactions_enriched.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

C:\Users\محمد الرويلي\AppData\Local\Temp\ipykernel_19284\1191469688.py:4: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/transactions_enriched.csv")


In [43]:
#02 — Get All Unique Dates
dates = pd.DataFrame({
    "FullDate": df["InvoiceDate"].dt.normalize().unique()
})

dates = dates.sort_values("FullDate").reset_index(drop=True)

In [44]:
#03 — Create the DateKey
dates["DateKey"] = (
    dates["FullDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [45]:
#04 — Generate Calendar Attributes
dates["Year"] = dates["FullDate"].dt.year
dates["Quarter"] = dates["FullDate"].dt.quarter
dates["Month"] = dates["FullDate"].dt.month
dates["MonthName"] = dates["FullDate"].dt.month_name()
dates["Week"] = dates["FullDate"].dt.isocalendar().week.astype(int)
dates["Day"] = dates["FullDate"].dt.day
dates["DayName"] = dates["FullDate"].dt.day_name()

In [46]:
#05 — Useful Business Flags
dates["IsWeekend"] = dates["FullDate"].dt.dayofweek >= 5

dates["IsMonthEnd"] = dates["FullDate"].dt.is_month_end

dates["IsQuarterEnd"] = dates["FullDate"].dt.is_quarter_end

dates["IsYearEnd"] = dates["FullDate"].dt.is_year_end

In [47]:
#06 — Reorder the Columns
dim_date = dates[
    [
        "DateKey",
        "FullDate",
        "Year",
        "Quarter",
        "Month",
        "MonthName",
        "Week",
        "Day",
        "DayName",
        "IsWeekend",
        "IsMonthEnd",
        "IsQuarterEnd",
        "IsYearEnd",
    ]
]

In [48]:
#07 — Check the Result
dim_date.head()

,DateKey,FullDate,Year,Quarter,Month,MonthName,Week,Day,DayName,IsWeekend,IsMonthEnd,IsQuarterEnd,IsYearEnd
0,20101201,2010-12-01,2010,4,12,December,48,1,Wednesday,False,False,False,False
1,20101202,2010-12-02,2010,4,12,December,48,2,Thursday,False,False,False,False
2,20101203,2010-12-03,2010,4,12,December,48,3,Friday,False,False,False,False
3,20101205,2010-12-05,2010,4,12,December,48,5,Sunday,True,False,False,False
4,20101206,2010-12-06,2010,4,12,December,49,6,Monday,False,False,False,False


In [49]:
#08 - save the Dimension
dim_date.to_csv(
    "../data/warehouse/DimDate.csv",
    index=False
)

Section 4 — DimCountry

In [50]:
#01 — Get Unique Countries
dim_country = (
    df[["Country"]]
    .drop_duplicates()
    .sort_values("Country")
    .reset_index(drop=True)
)

In [51]:
#02 — Create CountryKey
dim_country.insert(
    0,
    "CountryKey",
    range(1, len(dim_country) + 1)
)

In [52]:
#03 — Region Mapping
REGION_MAP = {
    "United Kingdom": "Northern Europe",
    "EIRE": "Northern Europe",
    "France": "Western Europe",
    "Germany": "Western Europe",
    "Netherlands": "Western Europe",
    "Belgium": "Western Europe",
    "Switzerland": "Central Europe",
    "Austria": "Central Europe",
    "Spain": "Southern Europe",
    "Portugal": "Southern Europe",
    "Italy": "Southern Europe",
    "Norway": "Northern Europe",
    "Sweden": "Northern Europe",
    "Denmark": "Northern Europe",
    "Finland": "Northern Europe",
    "Iceland": "Northern Europe",
    "Poland": "Eastern Europe",
    "Czech Republic": "Eastern Europe",
    "Lithuania": "Eastern Europe",
    "Greece": "Southern Europe",
    "Cyprus": "Southern Europe",
    "Malta": "Southern Europe",
    "Japan": "East Asia",
    "Singapore": "Southeast Asia",
    "Australia": "Oceania",
    "Israel": "Middle East",
    "United Arab Emirates": "Middle East",
    "Saudi Arabia": "Middle East",
    "Bahrain": "Middle East",
    "Lebanon": "Middle East",
    "Canada": "North America",
    "USA": "North America",
    "Brazil": "South America"
}

#Apply it
dim_country["Region"] = (
    dim_country["Country"]
    .map(REGION_MAP)
    .fillna("Other")
)

In [53]:
#04 — Continent Mapping
CONTINENT_MAP = {
    "Northern Europe": "Europe",
    "Western Europe": "Europe",
    "Central Europe": "Europe",
    "Southern Europe": "Europe",
    "Eastern Europe": "Europe",
    "Middle East": "Asia",
    "East Asia": "Asia",
    "Southeast Asia": "Asia",
    "North America": "North America",
    "South America": "South America",
    "Oceania": "Oceania"
}

#Apply it 
dim_country["Continent"] = (
    dim_country["Region"]
    .map(CONTINENT_MAP)
    .fillna("Other")
)

In [54]:
#05 — Market
MARKET_MAP = {
    "Europe": "EMEA",
    "Asia": "EMEA",
    "North America": "North America",
    "South America": "Latin America",
    "Oceania": "APAC"
}

#Apply it
dim_country["Market"] = (
    dim_country["Continent"]
    .map(MARKET_MAP)
    .fillna("Other")
)

In [55]:
#06 — Currency
CURRENCY_MAP = {
    "United Kingdom": "GBP",
    "EIRE": "EUR",
    "France": "EUR",
    "Germany": "EUR",
    "Spain": "EUR",
    "Portugal": "EUR",
    "Italy": "EUR",
    "Netherlands": "EUR",
    "Belgium": "EUR",
    "Austria": "EUR",
    "Finland": "EUR",
    "Greece": "EUR",
    "Cyprus": "EUR",
    "Malta": "EUR",
    "Lithuania": "EUR",
    "Poland": "PLN",
    "Switzerland": "CHF",
    "Norway": "NOK",
    "Sweden": "SEK",
    "Denmark": "DKK",
    "Japan": "JPY",
    "Australia": "AUD",
    "Canada": "CAD",
    "USA": "USD",
    "Israel": "ILS",
    "Singapore": "SGD",
    "Brazil": "BRL",
    "Saudi Arabia": "SAR",
    "United Arab Emirates": "AED",
    "Bahrain": "BHD",
    "Lebanon": "LBP"
}

#apply it
dim_country["Currency"] = (
    dim_country["Country"]
    .map(CURRENCY_MAP)
    .fillna("Unknown")
)

In [56]:
#07 — Reorder Columns
dim_country = dim_country[
    [
        "CountryKey",
        "Country",
        "Region",
        "Continent",
        "Market",
        "Currency"
    ]
]

In [57]:
#08 - Check
dim_country.head()

,CountryKey,Country,Region,Continent,Market,Currency
0,1,Australia,Oceania,Oceania,APAC,AUD
1,2,Austria,Central Europe,Europe,EMEA,EUR
2,3,Bahrain,Middle East,Asia,EMEA,BHD
3,4,Belgium,Western Europe,Europe,EMEA,EUR
4,5,Brazil,South America,South America,Latin America,BRL


In [58]:
#09 — Save Dimension
dim_country.to_csv(
    "../data/warehouse/DimCountry.csv",
    index=False
)